In [0]:
import re

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.dropdown(
    "schema",
    "migration_dev",
    ["migration_dev", "migration_test", "migration_prod_sim"],
)
dbutils.widgets.text("batch_id", "batch_001")

catalog_name = dbutils.widgets.get("catalog").strip()
schema_name = dbutils.widgets.get("schema").strip()
batch_id = dbutils.widgets.get("batch_id").strip()

if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", catalog_name):
    raise ValueError("Invalid catalog name")

if schema_name not in {
    "migration_dev", "migration_test", "migration_prod_sim"
}:
    raise ValueError("Invalid environment schema")

if not re.fullmatch(r"batch_[0-9]+", batch_id):
    raise ValueError("Invalid batch ID")

table_prefix = f"{catalog_name}.{schema_name}"
volume_root = f"/Volumes/{catalog_name}/{schema_name}"

print(f"Environment: {table_prefix}")
print(f"Batch: {batch_id}")
print(f"Volume root: {volume_root}")    

In [0]:
import hashlib
from pathlib import Path
from pyspark.sql import functions as F
 

controls = (
    spark.table(f"{table_prefix}.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(controls) != 1:
    raise RuntimeError("Expected one batch control record")

control = controls[0]

if control["batch_status"] != "APPROVED" or control["failed_checks"] != 0:
    raise RuntimeError("Batch is not approved")

# Inspect the latest outcome, rather than an older successful run.
latest = (
    spark.table(f"{table_prefix}.downstream_reconciliation")
    .filter(F.col("batch_id") == batch_id)
    .orderBy(F.col("reconciled_at").desc())
    .limit(1)
    .collect()
)

if not latest:
    raise RuntimeError("No downstream reconciliation exists")

reconciliation = latest[0]

if (
    reconciliation["load_completion_status"] != "COMPLETE"
    or reconciliation["validation_run_id"] != control["validation_run_id"]
):
    raise RuntimeError("Current approval has not completed reconciliation")

gold = (
    spark.table(f"{table_prefix}.gold_load_manifest")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

expected = control["expected_document_count"]

if not 0 < len(gold) == expected <= 1000:
    raise RuntimeError("Unexpected count or practice batch size exceeded")

for key in ["document_id", "file_name"]:
    if len({row[key] for row in gold}) != len(gold):
        raise RuntimeError(f"Duplicate {key}")

if any(
    row["validation_run_id"] != control["validation_run_id"]
    for row in gold
):
    raise RuntimeError("Gold approval is outdated")

downstream = Path(f"{volume_root}/downstream/{batch_id}")
archive = Path(f"{volume_root}/archive/{batch_id}")
archive.mkdir(parents=True, exist_ok=True)

copied = 0
verified = 0

for row in gold:
    filename = row["file_name"]

    if not filename or Path(filename).name != filename:
        raise RuntimeError("Unsafe filename")

    source = downstream / filename
    destination = archive / filename
    content = source.read_bytes()

    if hashlib.sha256(content).hexdigest() != row["source_sha256"]:
        raise RuntimeError(f"Downstream checksum mismatch: {filename}")

    if not destination.exists():
        with destination.open("xb") as output:
            output.write(content)
        copied += 1

    archive_hash = hashlib.sha256(destination.read_bytes()).hexdigest()

    if archive_hash != row["source_sha256"]:
        raise RuntimeError(f"Archive checksum mismatch: {filename}")

    verified += 1

print(f"New archive files copied: {copied}")
print(f"Archive files verified: {verified}")